# 02 · LoRA Fine-Tuning

Parameter-efficient fine-tuning with **LoRA** using Hugging Face `peft` + `trl`.
Theory: [docs/04_lora_explained.md](../docs/04_lora_explained.md).

Runs on a small model on CPU/GPU; swap the model name for a 7B on a GPU.


## 1. Install


In [ ]:
!pip install -q transformers datasets accelerate peft trl


## 2. Load base model + tokenizer


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # small but real
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto')
model.config.use_cache = False


## 3. Attach a LoRA adapter
Only `A` and `B` matrices on the attention projections will train. Watch the trainable %.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type='CAUSAL_LM')
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()  # e.g. 'trainable params: 0.06%'


## 4. Format the dataset (Alpaca style)
We reuse the repo's sample data and the same template as `src/data_prep.py`.


In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files='../data/sample_instructions.jsonl', split='train')
EOS = tokenizer.eos_token

def to_text(ex):
    inp = f"\n\n### Input:\n{ex['input']}" if ex['input'] else ''
    return {'text': ('Below is an instruction that describes a task. '
        'Write a response that appropriately completes the request.\n\n'
        f"### Instruction:\n{ex['instruction']}{inp}\n\n### Response:\n{ex['output']}" + EOS)}
ds = ds.map(to_text, remove_columns=ds.column_names)
print(ds[0]['text'])


## 5. Train with TRL's SFTTrainer


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

args = TrainingArguments(output_dir='out_lora', num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    logging_steps=5, fp16=True, report_to='none')
trainer = SFTTrainer(model=peft_model, args=args, train_dataset=ds,
    dataset_text_field='text', max_seq_length=512, tokenizer=tokenizer)
trainer.train()


## 6. Save and test the adapter
The saved adapter is only a few MB.


In [ ]:
trainer.save_model('lora-adapter')

prompt = ('Below is an instruction that describes a task. Write a response '
  'that appropriately completes the request.\n\n### Instruction:\n'
  'Explain what LoRA is in one sentence.\n\n### Response:\n')
inputs = tokenizer(prompt, return_tensors='pt').to(peft_model.device)
out = peft_model.generate(**inputs, max_new_tokens=80, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True).split('### Response:')[-1])


## Takeaways
- LoRA trained a fraction of a percent of parameters but adapts behavior.
- The adapter is tiny and swappable; merge it later with `src/merge_adapter.py`.
- Next: [03_qlora_mistral_peft.ipynb](03_qlora_mistral_peft.ipynb) scales this to a 7B model in 4-bit.
